In [1]:
import os
import time
import requests
import pandas as pd
from dotenv import load_dotenv
from pytrends.request import TrendReq

load_dotenv(".env")
TMDB_TOKEN = os.getenv("TMDB_TOKEN")

BASE_URL = "https://api.themoviedb.org/3"
GEO      = "DE"

tmdb_headers = {
    "accept": "application/json",
    "Authorization": f"Bearer {TMDB_TOKEN}"
}

pytrends = TrendReq(hl="de-DE", tz=60)

## Getter

In [ ]:
# LLM for retrying the collectionprocess
def tmdb_get(endpoint, params=None, retries=3):
    if params is None:
        params = {}
    url = BASE_URL + endpoint
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=tmdb_headers, params=params, timeout=10)
            r.raise_for_status()
            return r.json()
        except requests.exceptions.Timeout:
            print(f"Timeout, retry {attempt + 1}/{retries}...")
            time.sleep(2)
        except requests.exceptions.RequestException as e:
            print(f"Fehler: {e}")
            time.sleep(2)
    return None

## Collect Movies

In [3]:
def collect_movies(year, pages=5):
    all_movies = []
    for page in range(1, pages + 1):
        data = tmdb_get(
            "/discover/movie",
            params={
                "primary_release_year": year,
                "sort_by":              "popularity.desc",
                "vote_count.gte":       500,
                "page":                 page,
            }
        )
        all_movies.extend(data["results"])
        time.sleep(0.25)
    return pd.DataFrame(all_movies)

frames = []
for year in range(2010, 2024):
    frames.append(collect_movies(year, pages=5))
    print(f"Jahr {year} geladen")

df_movies = pd.concat(frames, ignore_index=True).drop_duplicates("id")
print(f"Gesamt: {len(df_movies)} Filme")

Jahr 2010 geladen
Jahr 2011 geladen
Jahr 2012 geladen
Jahr 2013 geladen
Jahr 2014 geladen
Jahr 2015 geladen
Jahr 2016 geladen
Jahr 2017 geladen
Jahr 2018 geladen
Jahr 2019 geladen
Jahr 2020 geladen
Jahr 2021 geladen
Jahr 2022 geladen
Jahr 2023 geladen
Gesamt: 1400 Filme


## Load Details

### Trends before

In [4]:
def get_trends_before(title, release):
    start = (release - pd.DateOffset(months=6)).strftime("%Y-%m-%d")
    end   = release.strftime("%Y-%m-%d")
    try:
        pytrends.build_payload(kw_list=[title], timeframe=f"{start} {end}", geo=GEO)
        df = pytrends.interest_over_time()
        return {
            "avg_trend_before":  round(df[title].mean(), 2) if not df.empty else None,
            "max_trend_before":  df[title].max()            if not df.empty else None,
            "peak_date_before":  df[title].idxmax()         if not df.empty else None,
        }
    except:
        return {"avg_trend_before": None, "max_trend_before": None, "peak_date_before": None}

### Trends After

In [5]:
def get_trends_after(title, release):
    start = release.strftime("%Y-%m-%d")
    end   = (release + pd.DateOffset(months=1)).strftime("%Y-%m-%d")
    try:
        pytrends.build_payload(kw_list=[title], timeframe=f"{start} {end}", geo=GEO)
        df = pytrends.interest_over_time()
        return {
            "avg_trend_after": round(df[title].mean(), 2) if not df.empty else None,
            "max_trend_after": df[title].max()            if not df.empty else None,
        }
    except:
        return {"avg_trend_after": None, "max_trend_after": None}

## Load Details

In [ ]:
rows = []

for _, movie in df_movies.iterrows():
    title   = movie["title"]
    release = pd.to_datetime(movie["release_date"])

    details = tmdb_get(f"/movie/{movie['id']}")
    if details is None:
        continue

    budget  = details.get("budget",  0)
    revenue = details.get("revenue", 0)

    if budget == 0 or revenue == 0:
        print(f"  skipped: {title}")
        continue

    before = get_trends_before(title, release)
    time.sleep(1)
    after  = get_trends_after(title, release)
    time.sleep(1)

    rows.append({
        "title":        title,
        "release_date": details.get("release_date"),
        "budget":       budget,
        "revenue":      revenue,
        "roi":          round(revenue / budget, 2),
        "vote_average": details.get("vote_average"),
        "vote_count":   details.get("vote_count"),
        "runtime":      details.get("runtime"),
        "genres":       ", ".join(g["name"] for g in details.get("genres", [])),
        **before,
        **after,
    })

    #print(f"Geladen: {title}")

print(f"Movies: {len(rows)}")

Geladen: Inception
  skipped: A Serbian Film
Geladen: Tangled
Geladen: Shrek Forever After
  skipped: Justice League: Crisis on Two Earths
Geladen: Shutter Island
Geladen: How to Train Your Dragon
Geladen: Legion
Geladen: Wall Street: Money Never Sleeps
Geladen: Harry Potter and the Deathly Hallows: Part 1
  skipped: Three Steps Above Heaven
Geladen: Percy Jackson & the Olympians: The Lightning Thief
Geladen: Toy Story 3
Geladen: The Expendables
Geladen: Black Swan
Geladen: Grown Ups
Geladen: Unstoppable
Geladen: Stake Land
Geladen: Iron Man 2
Geladen: Alice in Wonderland
Geladen: Despicable Me
Geladen: Predators
Geladen: TRON: Legacy
Geladen: The Fighter
Geladen: Due Date
  skipped: The Experiment
Geladen: True Grit
Geladen: Don't Be Afraid of the Dark
  skipped: Camp Rock 2: The Final Jam
Geladen: The Runaways
Geladen: Clash of the Titans
Geladen: The Wolfman
Geladen: Flipped
Geladen: You Will Meet a Tall Dark Stranger
Geladen: Charlie St. Cloud
Geladen: Megamind
Geladen: Salt
Gelade

## Store

In [ ]:
df = pd.DataFrame(rows)
df.to_csv("Q9.csv", index=False)
print(f"stored: {len(df)} Movies")
df.head()

Gespeichert: 1070 Filme


,title,release_date,budget,revenue,roi,vote_average,vote_count,runtime,genres,avg_trend_before,max_trend_before,peak_date_before,avg_trend_after,max_trend_after
0,Inception,2010-07-15,160000000,839030630,5.24,8.371,38810,148,"Action, Science Fiction, Adventure",NaN,NaN,NaT,NaN,NaN
1,Tangled,2010-11-24,260000000,592461732,2.28,7.609,12231,100,"Animation, Family, Adventure",NaN,NaN,NaT,NaN,NaN
2,Shrek Forever After,2010-05-20,165000000,752600867,4.56,6.394,7846,93,"Comedy, Adventure, Fantasy, Animation, Family",NaN,NaN,NaT,3.12,100.0
3,Shutter Island,2010-02-14,80000000,294804195,3.69,8.199,25508,138,"Drama, Thriller, Mystery",NaN,NaN,NaT,39.00,100.0
4,How to Train Your Dragon,2010-03-18,165000000,495141736,3.00,7.855,14158,98,"Fantasy, Adventure, Animation, Family",NaN,NaN,NaT,NaN,NaN
